# **Soliton Solutions to the Korteweg–De Vries (KdV) equation**

Here presents the differences in functionality between the legacy KdV PINN and the current refactored version

**Setup:**

In [1]:
import torch
import torch.nn as nn

import sys
from pathlib import Path

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from models import KDV

Check GPU availability:

In [2]:
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

CUDA available: True


# One Soliton

Configure and train a model for a single soliton, showing model initialization, training, testing, solution computation and plotting

In [6]:
INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 72, 
    verbose                  = False,
)

TRAIN_PARAMS = dict(
    adam_epochs              = 1000,
    verbose_step             = 100,
    n_collocation            = 50000, 
    n_initial                = 50000,  
    n_boundary               = 10000,
    n_momentum               = 30, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
    n_energy                 = 0,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
    adam_lr                  = 1e-3,   
    lbfgs_lr                 = 1.0,    
    lbfgs_history_size       = 100, 
    lbfgs_epochs             = 700,
    lbfgs_version            = 'new', #test is 'old' and anything else will default to a modified version of 'new' from legacy
    adaptive_sampling        = True,   
    logging                  = True, #new parameter, stops loss logging bottleneck for quick training (no loss history)
)

TRAIN_WEIGHTS = dict( #seperated out from the train params
    w_ic                     = 5.0,    
    w_bc                     = 2.0,    
    w_pde                    = 15.0,
    w_momentum               = 0.1,
    w_energy                 = 0.1
)

Create an instance of the class:

In [7]:
model = KDV(INIT_PARAMS)

Train the model:

In [8]:
training_stats, domain = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS) #renamed to not conflict with nn.Module superclass allows other integrations

Weighted losses [start]: IC=3.540e-01 | BC=1.685e-01 | PDE=8.985e-02
Starting Adam optimization...
[gpu mem] train start               alloc  1663.8 MB  reserved  3622.0 MB  peak  1663.8 MB
Adam - Epoch 0/1000, Total Loss: 3.577804e+00
Adam - Epoch 100/1000, Total Loss: 3.151689e-02
Adam - Epoch 200/1000, Total Loss: 1.702851e-02
Adam - Epoch 300/1000, Total Loss: 1.186138e-02
Adam - Epoch 400/1000, Total Loss: 9.430815e-03
Adam - Epoch 500/1000, Total Loss: 7.546947e-03
Adam - Epoch 600/1000, Total Loss: 5.912897e-03
Adam - Epoch 700/1000, Total Loss: 4.569252e-03
Adam - Epoch 800/1000, Total Loss: 3.583590e-03
Adam - Epoch 900/1000, Total Loss: 2.914984e-03
Adam - Epoch 999/1000, Total Loss: 2.458515e-03
[gpu mem] after Adam                alloc    21.1 MB  reserved  3808.0 MB  peak  3504.5 MB
Performing adaptive sampling...
Collocation points from 50000 -> 100000

Starting L-BFGS optimization...
L-BFGS - Iteration 1/700, Total Loss: 5.939365e-03
L-BFGS - Iteration 101/700, Total Los

KeyboardInterrupt: 

Plot the losses using the new returned dataclasses:  
(This was done so that loss histories can be saved and plotted again later much more easily)

In [ ]:
plot_loss = model.plot_losses(training_stats, components=('pde', 'total', 'momentum'))

Plot the profiles (Automatic Run):

In [ ]:
plot_auto_prof = model.plot_profiles(t_values=[-15, 0, 15], which=('predicted', 'exact')) #visualization functions now use OOP, returning a savable figure

Plot the profiles (Precomputed Solution):

In [ ]:
#UNDERCONSTRUCTION

Plot the spacetime (Automatic Run):

In [ ]:
plot_auto_st = model.plot_spacetime()

Plot the spacetime w/ scatter (Automatic Run):

In [ ]:
plot_auto_sst = model.plot_spacetime(scatter_which=('pde', 'boundary', 'initial'), training_domain=domain)

Plot the spacetime (Precomputed Solution):

In [ ]:
#UNDERCONSTRUCTION

Plot the spacetime w/ scatter (Precomputed Solution):

In [ ]:
#UNDERCONSTRUCTION

Now that the solutions have been visualized, test how accurate they are:  
The error type `absolute` and `absolute-normalized` are retained

In [ ]:
error_stats = model.test()

The plot heat map function is now a separate call

In [ ]:
plot_err = model.plot_heatmap()